# SEVA AI — Retrieval Pipeline Testing

This notebook tests the discovery and detail retrieval pipelines
using the indexed Chroma vector stores and parent documents.

Objectives:
- Test semantic retrieval.
- Identify relevant government schemes.
- Retrieve complete parent sections.
- Validate retrieval quality and performance.

In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [2]:
OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


In [3]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


## Step 2 — Load Existing Chroma Vector Stores

Connect to the previously indexed discovery and detail collections.
Verify that both collections contain the expected number of chunks.

In [4]:
discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


## Step 3 — Load Parent Documents

Load the complete parent sections from parent_documents.jsonl.

Create a lookup dictionary using parent_id so that retrieved child chunks
can be mapped back to their complete source sections.

In [5]:
parent_store = {}

with open(PARENT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parent = json.loads(line)
        parent_store[parent["parent_id"]] = parent

print("Parent documents loaded:", len(parent_store))
print("Unique parent IDs:", len(parent_store))

Parent documents loaded: 76366
Unique parent IDs: 76366


## Step 4 — Validate Child-to-Parent Relationships

Verify that every child chunk has a valid parent_id and a unique chunk_id.
This ensures that retrieved chunks can be mapped to complete parent sections.

In [7]:
child_paths = {
    "discovery": OUTPUT_DIR1 / "discovery_children.jsonl",
    "detail": OUTPUT_DIR1 / "detail_children.jsonl"
}

chunk_ids = set()
missing_parents = 0
total_chunks = 0

for index_name, file_path in child_paths.items():
    count = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            child = json.loads(line)

            total_chunks += 1
            count += 1

            chunk_ids.add(child["chunk_id"])

            if child["parent_id"] not in parent_store:
                missing_parents += 1

    print(f"{index_name.title()} chunks:", count)

print("\nTotal chunks:", total_chunks)
print("Unique chunk IDs:", len(chunk_ids))
print("Missing parent references:", missing_parents)

Discovery chunks: 23438
Detail chunks: 67945

Total chunks: 91383
Unique chunk IDs: 91383
Missing parent references: 0


## Step 5 — Test Semantic Discovery Retrieval

Search the discovery collection using a citizen's natural-language query.

Retrieve the top 10 relevant chunks with similarity scores and inspect
their scheme IDs, section types, and content.

In [16]:
query = (
    "I am a farmer and need financial assistance "
    "for crop cultivation. Which government schemes can help me?"
)

start = time.perf_counter()

results = discovery_db.similarity_search_with_score(
    query=query,
    k=30
)

latency = time.perf_counter() - start

print(f"Retrieved chunks: {len(results)}")
print(f"Retrieval latency: {latency:.3f} seconds")

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Rank: {rank}")
    print(f"Scheme ID: {doc.metadata.get('scheme_id')}")
    print(f"Section: {doc.metadata.get('child_type')}")
    print(f"Distance: {score:.4f}")
    print(f"Text: {doc.page_content[:300]}")

    

Retrieved chunks: 30
Retrieval latency: 0.302 seconds

Rank: 1
Scheme ID: sBe5F5gB5uF80_MGwZCt
Section: beneficiaries
Distance: 0.6145
Text: Scheme: Financial Assistance Scheme for Purchase of Agricultural Land to Scheduled Caste Farmers
Section: beneficiaries
Individual

Rank: 2
Scheme ID: sBe5F5gB5uF80_MGwZCt
Section: overview
Distance: 0.6227
Text: Scheme: Financial Assistance Scheme for Purchase of Agricultural Land to Scheduled Caste Farmers
Section: overview
The scheme was launched on in the year 1986. Under the scheme, financial assistance is provided to farmers belonging to the scheduled castes to purchase agricultural land, with a maximu

Rank: 3
Scheme ID: sBe5F5gB5uF80_MGwZCt
Section: benefits
Distance: 0.6441
Text: Scheme: Financial Assistance Scheme for Purchase of Agricultural Land to Scheduled Caste Farmers
Section: benefits
1. Under the scheme, Scheduled caste farmers are provided with financial assistance of ₹1,00,000/- per acre to purchase agricultural land.
 1. Assis

## Step 6 — Aggregate Retrieval Results by Scheme

Group retrieved discovery chunks by scheme_id.

For each scheme:
- Calculate its best retrieval distance.
- Count retrieved chunks.
- Collect unique section types.
- Preserve the best-matching chunk as evidence.

Rank schemes using their best distance.

In [ ]:
grouped_results = defaultdict(list)

for doc, distance in results:
    scheme_id = doc.metadata["scheme_id"]

    grouped_results[scheme_id].append({
        "distance": distance,
        "section": doc.metadata["child_type"],
        "text": doc.page_content
    })

scheme_results = []

for scheme_id, chunks in grouped_results.items():
    best_chunk = min(chunks, key=lambda x: x["distance"])

    scheme_results.append({
        "scheme_id": scheme_id,
        "best_distance": best_chunk["distance"],
        "chunk_count": len(chunks),
        "sections": sorted(set(c["section"] for c in chunks)),
        "best_evidence": best_chunk["text"]
    })

scheme_results.sort(key=lambda x: x["best_distance"])

for rank, scheme in enumerate(scheme_results, start=1):
    print(f"\nRank: {rank}")
    print("Scheme ID:", scheme["scheme_id"])
    print("Best distance:", round(scheme["best_distance"], 4))
    print("Retrieved chunks:", scheme["chunk_count"])
    print("Sections:", scheme["sections"])
    print("Evidence:", scheme["best_evidence"][:250])
    print("-" * 60)

print("\nUnique schemes:", len(scheme_results))


Rank: 1
Scheme ID: sBe5F5gB5uF80_MGwZCt
Best distance: 0.6145
Retrieved chunks: 4
Sections: ['beneficiaries', 'benefits', 'eligibility', 'overview']
Evidence: Scheme: Financial Assistance Scheme for Purchase of Agricultural Land to Scheduled Caste Farmers
Section: beneficiaries
Individual
------------------------------------------------------------

Rank: 2
Scheme ID: gsbnF5gB0AGc0F0iWDfo
Best distance: 0.6591
Retrieved chunks: 1
Sections: ['eligibility']
Evidence: 1. Crop assistance will be provided to farmers only in case of crop loss of at least 20%.
 1. This scheme will be voluntary for all farmers.
------------------------------------------------------------

Rank: 3
Scheme ID: oKGsTpoBXu-iUKXqG3oC
Best distance: 0.6663
Retrieved chunks: 3
Sections: ['benefits', 'eligibility', 'overview']
Evidence: Scheme: Investment Grant for Farmers with More than 2 Hectares of Agricultural Land
Section: overview
The scheme aims to provide financial assistance to farmers owning more than 2 hect

In [9]:
query = (
    "I am a farmer and need financial assistance "
    "for crop cultivation. Which government schemes can help me?"
)

start = time.perf_counter()

results_30 = discovery_db.similarity_search_with_score(
    query=query,
    k=30
)

latency_30 = time.perf_counter() - start

grouped_30 = defaultdict(list)

for doc, distance in results_30:
    grouped_30[doc.metadata["scheme_id"]].append({
        "distance": distance,
        "section": doc.metadata["child_type"],
        "text": doc.page_content
    })

scheme_results_30 = []

for scheme_id, chunks in grouped_30.items():
    best_chunk = min(chunks, key=lambda x: x["distance"])

    scheme_results_30.append({
        "scheme_id": scheme_id,
        "best_distance": best_chunk["distance"],
        "chunk_count": len(chunks),
        "sections": sorted(set(c["section"] for c in chunks))
    })

scheme_results_30.sort(key=lambda x: x["best_distance"])

print("Retrieved chunks:", len(results_30))
print("Unique schemes:", len(scheme_results_30))
print(f"Latency: {latency_30:.3f} seconds")

for rank, scheme in enumerate(scheme_results_30, start=1):
    print(
        f"\n{rank}. {scheme['scheme_id']} | "
        f"Distance: {scheme['best_distance']:.4f} | "
        f"Chunks: {scheme['chunk_count']} | "
        f"Sections: {scheme['sections']}"
    )

Retrieved chunks: 30
Unique schemes: 21
Latency: 0.388 seconds

1. sBe5F5gB5uF80_MGwZCt | Distance: 0.6145 | Chunks: 4 | Sections: ['beneficiaries', 'benefits', 'eligibility', 'overview']

2. gsbnF5gB0AGc0F0iWDfo | Distance: 0.6591 | Chunks: 1 | Sections: ['eligibility']

3. oKGsTpoBXu-iUKXqG3oC | Distance: 0.6663 | Chunks: 3 | Sections: ['benefits', 'eligibility', 'overview']

4. dRiBwpwB5uF80_MGrV-q | Distance: 0.6862 | Chunks: 1 | Sections: ['overview']

5. CsYxGJgB0AGc0F0i6jir | Distance: 0.6898 | Chunks: 1 | Sections: ['benefits']

6. K8bZF5gB0AGc0F0i0TfE | Distance: 0.6954 | Chunks: 2 | Sections: ['benefits', 'overview']

7. 6ca5F5gB0AGc0F0iSjSm | Distance: 0.7040 | Chunks: 2 | Sections: ['benefits', 'eligibility']

8. ihcuo5gB5uF80_MGvJTz | Distance: 0.7073 | Chunks: 1 | Sections: ['benefits']

9. tca3F5gB0AGc0F0iszQY | Distance: 0.7173 | Chunks: 1 | Sections: ['benefits']

10. MsY9GJgB0AGc0F0iBjji | Distance: 0.7237 | Chunks: 2 | Sections: ['benefits', 'eligibility']

11. 58czY

In [10]:
candidate_scheme_ids = [
    scheme["scheme_id"]
    for scheme in scheme_results_30
]

detail_query = (
    "What documents are required to apply for "
    "government financial assistance schemes for farmers?"
)

start = time.perf_counter()

detail_results = detail_db.similarity_search_with_score(
    query=detail_query,
    k=20,
    filter={
        "scheme_id": {"$in": candidate_scheme_ids}
    }
)

detail_latency = time.perf_counter() - start

print("Candidate schemes:", len(candidate_scheme_ids))
print("Retrieved detail chunks:", len(detail_results))
print(f"Detail retrieval latency: {detail_latency:.3f} seconds")

for rank, (doc, distance) in enumerate(detail_results, start=1):
    print(f"\n{'=' * 60}")
    print("Rank:", rank)
    print("Scheme ID:", doc.metadata.get("scheme_id"))
    print("Section:", doc.metadata.get("child_type"))
    print("Parent ID:", doc.metadata.get("parent_id"))
    print("Distance:", round(distance, 4))
    print("Text:", doc.page_content[:400])

Candidate schemes: 21
Retrieved detail chunks: 20
Detail retrieval latency: 0.278 seconds

Rank: 1
Scheme ID: LqE9GJgBXu-iUKXqBHm0
Section: faq
Parent ID: LqE9GJgBXu-iUKXqBHm0_FAQ_007
Distance: 0.5162
Text: Scheme: Scheme of Providing Assistance to Encourage Farmers for Adoption of Organic Farming
Question: What documents are required for the application process?
Answer: The required documents include an Aadhaar card, land ownership proof, bank account details, organic certification documents (if available) etc.

Rank: 2
Scheme ID: AqG_F5gBXu-iUKXq7ne9
Section: faq
Parent ID: AqG_F5gBXu-iUKXq7ne9_FAQ_009
Distance: 0.5272
Text: Scheme: Scheme to Encourage Farmers to Add Value to Crops
Question: What documents are required to apply for this scheme?
Answer: The required documents include an Aadhaar card, a 7/12 land record, and either a bank passbook or a cancelled cheque.

Rank: 3
Scheme ID: sBe5F5gB5uF80_MGwZCt
Section: documents_required
Parent ID: sBe5F5gB5uF80_MGwZCt_DOCUMENTS_REQUIR